# Stage11: Evaluation & Risk Communication

This notebook evaluates the fixed Stage10 baseline on its future-like test period; it does not retune the model on test outcomes.


In [1]:
from pathlib import Path
import os,sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
if Path.cwd().name=="notebooks": os.chdir("..")
ROOT=Path.cwd()
if not (ROOT/"src"/"modeling.py").is_file():
    for x in (ROOT,*ROOT.parents):
        if (x/"project"/"src"/"modeling.py").is_file(): ROOT=x/"project";break
sys.path.insert(0,str(ROOT))
from src.modeling import run_baseline
from src.evaluation import bootstrap_pr_auc,sensitivity_tables
from src.storage import read_df
P=ROOT/"data"/"processed";R=ROOT/"reports";source=read_df(P/"spy_feature_candidates_20260907-143336.parquet");result=run_baseline(source)


## 1. Bootstrap uncertainty, scenario sensitivity, and subgroup diagnostics


In [2]:
ci,boot=bootstrap_pr_auc(result["test"].label,result["test_probability"],n_boot=600,seed=111)
scenarios,subgroups=sensitivity_tables(result["test"],result["test_probability"],result["cutoff"])
ts="20260907-143336";pd.DataFrame([ci]).to_csv(P/f"bootstrap_pr_auc_{ts}.csv",index=False);scenarios.to_csv(P/f"evaluation_scenarios_{ts}.csv",index=False);subgroups.to_csv(P/f"evaluation_subgroups_{ts}.csv",index=False)
fig,ax=plt.subplots(1,3,figsize=(15,4));ax[0].hist(boot,bins=30,color="#2563EB");ax[0].axvline(ci["lower_95"],ls="--",color="black");ax[0].axvline(ci["upper_95"],ls="--",color="black");ax[0].set(title="Bootstrap PR-AUC",xlabel="PR-AUC");ax[1].bar(scenarios["scenario"],scenarios["recall"],color="#0F766E");ax[1].set(title="Cutoff scenario recall",ylim=(0,1));ax[1].tick_params(axis="x",rotation=20);ax[2].bar(subgroups["regime"],subgroups["recall"],color="#7C3AED");ax[2].set(title="Volatility-regime recall",ylim=(0,1));fig.tight_layout();plot=R/f"spy_evaluation_{ts}.png";fig.savefig(plot,dpi=150,bbox_inches="tight");display(pd.DataFrame([ci]));display(scenarios);display(subgroups)
assert len(boot)==600 and ci["lower_95"]<=ci["upper_95"] and plot.is_file();print("Stage11 checks passed.")


,estimate,lower_95,upper_95,samples
0,0.292812,0.179133,0.457212,600


,scenario,accuracy,precision,recall,f1,pr_auc,alert_rate,true_negative,false_positive,false_negative,true_positive
0,validation_f1_cutoff,0.858283,0.242857,0.485714,0.32381,0.292812,0.139721,413,53,18,17
1,conservative_0.70,0.908184,0.310345,0.257143,0.28125,0.292812,0.057884,446,20,26,9


,regime,rows,accuracy,precision,recall,f1,pr_auc,alert_rate,true_negative,false_positive,false_negative,true_positive
0,lower_volatility,251,0.940239,0.000000,0.000000,0.000000,0.041490,0.023904,236,6,9,0
1,higher_volatility,250,0.776000,0.265625,0.653846,0.377778,0.376844,0.256000,177,47,9,17


Stage11 checks passed.


## 2. Stakeholder summary

The baseline has limited ranking signal, but its bootstrap interval is wide and lower-volatility recall is zero. Use it only as a review trigger; monitor false negatives, alert rate, data quality, and regime changes.
